# TDT13 IMDb sentiment: Qwen runs on Colab (development only)

Runs the Qwen2.5-1.5B-Instruct part of the project on a Colab GPU, with the repository and all
outputs stored on Google Drive (`/content/drive/MyDrive/tdt13`).

**Before running:** choose **Runtime → Change runtime type → L4 GPU** (Colab Pro). The model
runs in bfloat16 (`llm.dtype`), which the L4 supports natively; a T4 does not and is much slower.
The notebook stops if no CUDA GPU is found and warns if the GPU is not an L4. All test
conditions must later run on the same GPU type.

**Resuming:** every LLM batch is appended to `runs/<split_tag>/<condition>/predictions.jsonl`
on Drive and fsync'ed. If the runtime disconnects, reconnect and choose **Runtime → Run all**.
Finished reviews are skipped, so the runs continue where they stopped. Data preparation and
baselines are skipped if their outputs already exist.

**Scope:** this notebook only uses the training and validation data. Everything it produces
is a **development result**. No cell runs `freeze` or touches the test split; test runs come
later, after `freeze`, in a separate notebook.

## Expected fingerprints

These values come from the local data preparation (`data/processed/data_manifest.json` →
`split_files`, and the SHA-256 of `data/processed/demo_sets.json`, which includes the
secondary `_flip` order conditions). The notebook stops if the
data on Drive does not match them, so Colab runs use exactly the same splits and demonstrations.

If they differ after a fresh `prepare` on Colab, do not edit these values: upload the local
`data/processed/` folder to `MyDrive/tdt13/data/processed/` instead and run all again.

In [ ]:
EXPECTED_SPLIT_FILES = {
    "train_fit": "7c9187b7047e6edfbf894920caecddb23962faee95102a0a4d6f5838c3ab735a",
    "validation": "0e51f1741c152da279c75e98354a8b1d41e124fe4f26858d1c88287bad74a914",
    "test": "0c76b0090c3c4d7f1d40955a93b25d05cd2b8efd913af04a674c065017168161",
}
EXPECTED_DEMO_SETS_SHA256 = "026458fefdc6230ff730c2c01bffd302209e63da4117939764d6bd34cba8ea52"
# demo_sets.json before the secondary _flip conditions were appended (original conditions
# identical). A Drive copy with this hash is regenerated once with select-demos.
PRE_FLIP_DEMO_SETS_SHA256 = "d883d9ea5a8034e50be44b0e62a0b3754d3c880d4b086d0d0774c3c3adccca76"

REPO_URL = "https://github.com/Lasalei/tdt13.git"
REPO_DIR = "/content/drive/MyDrive/tdt13"

## Helper and GPU check

In [ ]:
import os, subprocess, sys

def run(cmd, cwd=None):
    """Run a shell command, stream its output, and stop the notebook if it fails.

    PYTHONUNBUFFERED=1 makes the child Python process print progress immediately instead
    of buffering it until the command ends."""
    print(f"$ {cmd}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")

import torch
assert torch.cuda.is_available(), "No CUDA GPU. Use Runtime -> Change runtime type -> L4 GPU."
import transformers
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if "L4" not in GPU_NAME:
    print(f"WARNING: GPU is {GPU_NAME!r}, not an L4. The project runs on L4 "
          "(native bfloat16). Runs on other GPUs are slower and must not be mixed "
          "with L4 runs in the final comparison.")
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("transformers (preinstalled):", transformers.__version__)
print("python:", sys.version.split()[0])

## Mount Drive and get the code

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(f"git clone {REPO_URL} {REPO_DIR}")
else:
    run("git pull --ff-only", cwd=REPO_DIR)
os.chdir(REPO_DIR)
run("git log -1 --oneline")

## Install

Colab ships a CUDA build of torch. If it satisfies `torch>=2.2`, the `torch` line is dropped
from `requirements-llm.txt` so pip cannot replace it with a different build.

In [ ]:
from packaging.version import Version

torch_ok = Version(torch.__version__.split("+")[0]) >= Version("2.2") and torch.version.cuda is not None
llm_reqs = "requirements-llm.txt"
if torch_ok:
    lines = open(llm_reqs).read().splitlines()
    keep = [l for l in lines if not l.strip().lower().startswith("torch")]
    llm_reqs = "/content/requirements-llm-no-torch.txt"
    open(llm_reqs, "w").write("\n".join(keep) + "\n")
    print(f"Keeping Colab torch {torch.__version__}")
run(f"pip install -q -r requirements.txt -r {llm_reqs}")
run("pip install -q -e .")
run('python -c "import torch, transformers; print(\'torch\', torch.__version__, \'cuda\', torch.cuda.is_available()); print(\'transformers\', transformers.__version__)"')

If pip upgraded a package that was already imported (for example `transformers`), Colab may ask
you to restart the session. Do so, then **Run all** again; the install step is then a no-op.

## Technical tests (synthetic data)

In [ ]:
run("python -m pytest -q")

## Data and demonstrations (only if missing)

If `demo_sets.json` on Drive is the version from before the `_flip` order conditions were
added, it is regenerated once with `select-demos`. Selection is deterministic, so the
original conditions stay identical and the `_flip` conditions are appended.

In [ ]:
import hashlib

if not os.path.isfile("data/processed/data_manifest.json"):
    run("python -m imdbsent prepare")
else:
    print("data/processed exists; skipping prepare")
demo_path = "data/processed/demo_sets.json"
if not os.path.isfile(demo_path):
    run("python -m imdbsent select-demos")
elif hashlib.sha256(open(demo_path, "rb").read()).hexdigest() == PRE_FLIP_DEMO_SETS_SHA256:
    print("demo_sets.json predates the _flip conditions; regenerating")
    run("python -m imdbsent select-demos")
else:
    print("demo_sets.json exists; skipping select-demos")

## Verify fingerprints

In [ ]:
import hashlib, json

manifest = json.load(open("data/processed/data_manifest.json"))
demo_sha = hashlib.sha256(open("data/processed/demo_sets.json", "rb").read()).hexdigest()

mismatch = []
print(f"{'file':<16}{'found':<66}expected")
for split, exp in EXPECTED_SPLIT_FILES.items():
    got = manifest["split_files"].get(split)
    print(f"{split:<16}{str(got):<66}{exp}")
    if got != exp:
        mismatch.append(split)
print(f"{'demo_sets.json':<16}{demo_sha:<66}{EXPECTED_DEMO_SETS_SHA256}")
if demo_sha != EXPECTED_DEMO_SETS_SHA256:
    mismatch.append("demo_sets.json")
print("dataset revision:", manifest["provenance"].get("hf_revision_resolved"))

if mismatch:
    raise RuntimeError(
        f"Fingerprint mismatch for {mismatch}. The data on Drive differs from the local preparation. "
        "Upload the local data/processed/ folder to MyDrive/tdt13/data/processed/ and run all again.")
print("All fingerprints match.")

## Baselines on validation (CPU, ~2 min, only if missing)

`evaluate --split-tag validation-dev` scores VADER and TF-IDF+LR on the same 500 dev reviews as
the LLM, using the validation baseline runs. They are produced here if they are not on Drive.

In [ ]:
baseline_conds = ["vader", "vader_fulltext", "tfidf_lr", "tfidf_lr_fulltext"]
missing = [c for c in baseline_conds
           if not os.path.isfile(f"runs/validation/{c}/run_manifest.json")
           or not json.load(open(f"runs/validation/{c}/run_manifest.json")).get("complete")]
if missing:
    run("python -m imdbsent baselines --split validation --conditions " + " ".join(missing))
else:
    print("Validation baselines exist; skipping")

## Archive the earlier T4 dev runs (only once)

The first dev runs were made on a T4 with `llm.dtype: auto`. The config now pins `bfloat16`,
which changes the Qwen settings hash, so those runs cannot be resumed. If
`runs/validation-dev` holds runs made with different Qwen settings and no archive exists yet,
it is renamed to `runs/validation-dev-t4-autodtype` (and `results/validation-dev` to
`results/validation-dev-t4-autodtype`). They are kept as a record; the dev runs are then
redone below. Runs that match the current settings are never moved.

In [ ]:
import glob

ARCHIVE = "validation-dev-t4-autodtype"
current_hash = subprocess.check_output(
    [sys.executable, "-c", "from imdbsent.config import load_config; "
     "from imdbsent.predio import settings_hash; print(settings_hash(load_config(), 'qwen'))"],
    text=True).strip()
manifests = [json.load(open(f)) for f in sorted(glob.glob("runs/validation-dev/qwen_*/run_manifest.json"))]
stale = [m["condition"] for m in manifests if m.get("settings_hash") != current_hash]
print("current Qwen settings hash:", current_hash)

if os.path.exists(f"runs/{ARCHIVE}"):
    print(f"runs/{ARCHIVE} already exists; nothing to archive")
elif not os.path.isdir("runs/validation-dev"):
    print("no runs/validation-dev yet; nothing to archive")
elif not stale:
    print("runs/validation-dev matches the current settings; nothing to archive")
else:
    print("runs with older Qwen settings:", stale)
    os.rename("runs/validation-dev", f"runs/{ARCHIVE}")
    print(f"renamed runs/validation-dev -> runs/{ARCHIVE}")
    if os.path.isdir("results/validation-dev") and not os.path.exists(f"results/{ARCHIVE}"):
        os.rename("results/validation-dev", f"results/{ARCHIVE}")
        print(f"renamed results/validation-dev -> results/{ARCHIVE}")

## Pilot

48 balanced dev reviews × {zero-shot, 2-shot set 1, 4-shot set 1}, with the batch size from
`configs/experiment.yaml`. Checks the full Qwen path on the GPU and estimates the runtime.
Pilot runs always start from scratch (`runs/pilot/` is overwritten).

In [ ]:
run("python -m imdbsent pilot --n 48")
print(open("results/pilot_estimate_hf.json").read())

## Dev runs (500-review validation-dev subset, all 7 LLM conditions)

Resumable: if this cell is interrupted, reconnect and **Run all**. Afterwards, the evaluation
scores all LLM conditions and the baselines on the same 500 reviews. These are
**development results**.

In [ ]:
run("python -m imdbsent llm --condition all --split-tag validation-dev")

In [ ]:
run("python -m imdbsent evaluate --split-tag validation-dev")
import pandas as pd
print(open("results/validation-dev/STATUS.txt").read())
pd.set_option("display.width", 200)
pd.read_csv("results/validation-dev/metrics_summary.csv")

## Not in this notebook: test runs

Test-split runs come later, after the setup is frozen (`python -m imdbsent freeze`), in a
separate notebook. Nothing here runs `freeze` or loads the test split.